In [1]:
import rawpy
import numpy as np
import os
import csv
import traceback
from pathlib import Path
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
from PIL import Image, ImageTk
import threading
import subprocess
import platform

def extract_2x2_rgb_average(raw_file_path, x, y):
    """
    Extract the average RGB values of a 2x2 region at specified coordinates from a RAW image.
    
    Parameters:
        raw_file_path: Path to DNG file
        x, y: Coordinates (top-left corner) in the debayered RGB image coordinate system
    
    Returns:
        dict: Dictionary containing R, G, B averages and statistics
    """
    # Use context manager to read RAW file (recommended by rawpy)
    try:
        # Ensure the path is a string
        file_path_str = str(Path(raw_file_path).resolve())
        
        # Read RAW file with context manager
        with rawpy.imread(file_path_str) as raw:
            # Convert RAW image to RGB image (demosaic)
            # Use default parameters, keep original size
            rgb_array = raw.postprocess(
                use_camera_wb=True,      # Use camera white balance
                half_size=False,         # Do not half the size
                no_auto_bright=True,     # Do not auto-adjust brightness
                output_bps=16,           # 16-bit output for higher precision
                demosaic_algorithm=rawpy.DemosaicAlgorithm.AHD  # Use AHD algorithm
            )
            
            # rgb_array shape: (height, width, 3), channels: R, G, B
            height, width, channels = rgb_array.shape
            
            # Check if coordinates are within valid range (considering the 2x2 region)
            if x < 0 or y < 0 or x + 2 > width or y + 2 > height:
                raise ValueError(f"Coordinate ({x}, {y}) out of image range ({width}x{height})")
            
            # Extract the 2x2 region's RGB values
            # region shape: (2, 2, 3) containing 4 pixels, each with R,G,B
            region = rgb_array[y:y+2, x:x+2, :]
            
            # Extract each pixel's RGB values
            # Pixel positions:
            # [0,0] top-left
            # [0,1] top-right
            # [1,0] bottom-left
            # [1,1] bottom-right
            pixel_00 = region[0, 0, :]  # top-left pixel (R, G, B)
            pixel_01 = region[0, 1, :]  # top-right pixel (R, G, B)
            pixel_10 = region[1, 0, :]  # bottom-left pixel (R, G, B)
            pixel_11 = region[1, 1, :]  # bottom-right pixel (R, G, B)
            
            # Extract R, G, B values from all pixels
            r_values = [int(pixel_00[0]), int(pixel_01[0]), int(pixel_10[0]), int(pixel_11[0])]
            g_values = [int(pixel_00[1]), int(pixel_01[1]), int(pixel_10[1]), int(pixel_11[1])]
            b_values = [int(pixel_00[2]), int(pixel_01[2]), int(pixel_10[2]), int(pixel_11[2])]
            
            # Compute averages
            r_avg = float(np.mean(r_values))
            g_avg = float(np.mean(g_values))
            b_avg = float(np.mean(b_values))
            
            # Compute statistics
            r_min = float(np.min(r_values))
            r_max = float(np.max(r_values))
            g_min = float(np.min(g_values))
            g_max = float(np.max(g_values))
            b_min = float(np.min(b_values))
            b_max = float(np.max(b_values))
            
            r_std = float(np.std(r_values))
            g_std = float(np.std(g_values))
            b_std = float(np.std(b_values))
            
            result = {
                'file': os.path.basename(raw_file_path),
                'x': x,
                'y': y,
                'R': r_avg,
                'G': g_avg,
                'B': b_avg,
                'G2': g_avg,  # G2 is same as G (each pixel has full RGB after demosaic)
                'R_min': r_min,
                'R_max': r_max,
                'G_min': g_min,
                'G_max': g_max,
                'B_min': b_min,
                'B_max': b_max,
                'G2_min': g_min,  # G2 statistics same as G
                'G2_max': g_max,
                'R_std': r_std,
                'G_std': g_std,
                'B_std': b_std,
                'G2_std': g_std,  # G2 std same as G
                'pixel_values': {
                    'pixel_00': {'R': int(pixel_00[0]), 'G': int(pixel_00[1]), 'B': int(pixel_00[2])},
                    'pixel_01': {'R': int(pixel_01[0]), 'G': int(pixel_01[1]), 'B': int(pixel_01[2])},
                    'pixel_10': {'R': int(pixel_10[0]), 'G': int(pixel_10[1]), 'B': int(pixel_10[2])},
                    'pixel_11': {'R': int(pixel_11[0]), 'G': int(pixel_11[1]), 'B': int(pixel_11[2])}
                }
            }
            
            return result
                    
    except Exception as e:
        error_type = type(e).__name__
        error_msg = str(e)
        error_traceback = traceback.format_exc()
        print(f"Error processing file {raw_file_path}:")
        print(f"  Error type: {error_type}")
        print(f"  Error message: {error_msg}")
        print(f"  Traceback:\n{error_traceback}")
        return None

def process_image_folder(folder_path, x, y, output_csv='output.csv', progress_callback=None):
    """
    Process all DNG images in a folder.
    
    Parameters:
        folder_path: Path to folder containing DNG files
        x, y: Coordinates to extract
        output_csv: Output CSV file path (absolute path)
        progress_callback: Progress callback function receiving (current, total, filename)
    
    Returns:
        str: Full path of saved file
    """
    folder = Path(folder_path)
    # Use case-insensitive search for DNG files to avoid duplicates
    dng_files = sorted(set(folder.glob('*.DNG')) | set(folder.glob('*.dng')))
    
    if not dng_files:
        print(f"No DNG files found in folder {folder_path}")
        return None
    
    print(f"Found {len(dng_files)} DNG files")
    print(f"Processing 2x2 region at ({x}, {y})...")
    print("-" * 60)
    
    results = []
    failed_files = []
    total_files = len(dng_files)
    
    for idx, dng_file in enumerate(dng_files, 1):
        print(f"[{idx}/{total_files}] Processing: {dng_file.name}")
        
        # Update progress callback
        if progress_callback:
            progress_callback(idx, total_files, dng_file.name)
        
        result = extract_2x2_rgb_average(dng_file, x, y)
        if result:
            results.append(result)
            print(f"  ✓ Success")
        else:
            failed_files.append(dng_file.name)
            print(f"  ✗ Failed")
    
    print("-" * 60)
    print(f"Processing completed: {len(results)} succeeded, {len(failed_files)} failed")
    
    if failed_files:
        print(f"Failed files:")
        for fname in failed_files[:10]:  # Show only first 10
            print(f"  - {fname}")
        if len(failed_files) > 10:
            print(f"  ... and {len(failed_files) - 10} more failed")
    
    if not results:
        print("\nError: No files were successfully processed!")
        print("Please check:")
        print("1. Whether DNG files are corrupt")
        print("2. Whether coordinates are within image range")
        print("3. Whether rawpy library is correctly installed")
        return None
    
    # Ensure absolute path is used
    output_path = Path(output_csv).resolve()
    
    # Export to CSV
    if results:
        with open(output_path, 'w', newline='', encoding='utf-8-sig') as f:
            fieldnames = ['file', 'x', 'y', 'R', 'G', 'B', 'G2', 
                         'R_min', 'R_max', 'G_min', 'G_max', 'B_min', 'B_max', 
                         'G2_min', 'G2_max', 'R_std', 'G_std', 'B_std', 'G2_std']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for result in results:
                # Write only CSV fields, exclude pixel_values
                row = {k: v for k, v in result.items() if k != 'pixel_values'}
                writer.writerow(row)
        
        print(f"\nData exported to: {output_path}")
        
        # Compute average across all images
        if len(results) > 1:
            r_values = [r['R'] for r in results]
            g_values = [r['G'] for r in results]
            b_values = [r['B'] for r in results]
            g2_values = [r['G2'] for r in results]
            
            print(f"\nAverage statistics across all images:")
            print(f"R: mean={np.mean(r_values):.2f}, min={np.min(r_values):.2f}, max={np.max(r_values):.2f}, std={np.std(r_values):.2f}")
            print(f"G: mean={np.mean(g_values):.2f}, min={np.min(g_values):.2f}, max={np.max(g_values):.2f}, std={np.std(g_values):.2f}")
            print(f"B: mean={np.mean(b_values):.2f}, min={np.min(b_values):.2f}, max={np.max(b_values):.2f}, std={np.std(b_values):.2f}")
            print(f"G2: mean={np.mean(g2_values):.2f}, min={np.min(g2_values):.2f}, max={np.max(g2_values):.2f}, std={np.std(g2_values):.2f}")
    
    return str(output_path)

class CoordinateSelector:
    def __init__(self, folder_path, output_csv='rgb_average_results.csv'):
        self.folder_path = folder_path
        # Use absolute path, save in the script's directory (or current working directory if __file__ not defined)
        try:
            script_dir = Path(__file__).parent.resolve()
        except NameError:
            # In interactive environments (e.g., Jupyter, IPython) __file__ is not defined
            script_dir = Path.cwd()
        self.output_csv = str(script_dir / output_csv)
        self.output_file_path = None  # Store the actual generated file path
        self.selected_x = None
        self.selected_y = None
        self.original_image = None
        self.display_image = None
        self.scale_factor = 1.0
        self.base_scale_factor = 1.0  # Initial scaling ratio
        self.current_zoom = 1.0  # Current zoom level
        self.original_width = 0
        self.original_height = 0
        
        # Create main window
        self.root = tk.Tk()
        self.root.title("Select Coordinates - DNG Image Processing")
        self.root.geometry("1200x800")
        
        # Create main frame
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Configure grid weights
        self.root.columnconfigure(0, weight=1)
        self.root.rowconfigure(0, weight=1)
        main_frame.columnconfigure(0, weight=1)
        main_frame.rowconfigure(1, weight=1)  # Canvas row
        
        # Info label and folder display
        top_frame = ttk.Frame(main_frame)
        top_frame.grid(row=0, column=0, pady=(5, 2), sticky=(tk.W, tk.E))
        top_frame.columnconfigure(1, weight=1)
        
        info_label = ttk.Label(top_frame, text="Click on image to select coordinates (top-left corner of 2x2 region)", 
                              font=('Arial', 12))
        info_label.grid(row=0, column=0, columnspan=2, pady=(0, 2))
        
        folder_label_frame = ttk.Frame(top_frame)
        folder_label_frame.grid(row=1, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=0)
        folder_label_frame.columnconfigure(1, weight=1)
        
        ttk.Label(folder_label_frame, text="Current folder:", font=('Arial', 9)).grid(row=0, column=0, padx=2)
        self.folder_path_label = ttk.Label(folder_label_frame, text=folder_path, 
                                          font=('Arial', 9), foreground='blue')
        self.folder_path_label.grid(row=0, column=1, sticky=tk.W, padx=2)
        
        # Create canvas and scrollbars
        canvas_frame = ttk.Frame(main_frame)
        canvas_frame.grid(row=1, column=0, sticky=(tk.W, tk.E, tk.N, tk.S), pady=2)
        canvas_frame.columnconfigure(0, weight=1)
        canvas_frame.rowconfigure(0, weight=1)
        
        self.canvas = tk.Canvas(canvas_frame, bg='gray', cursor='crosshair')
        v_scrollbar = ttk.Scrollbar(canvas_frame, orient=tk.VERTICAL, command=self.canvas.yview)
        h_scrollbar = ttk.Scrollbar(canvas_frame, orient=tk.HORIZONTAL, command=self.canvas.xview)
        
        self.canvas.configure(yscrollcommand=v_scrollbar.set, xscrollcommand=h_scrollbar.set)
        
        self.canvas.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        v_scrollbar.grid(row=0, column=1, sticky=(tk.N, tk.S))
        h_scrollbar.grid(row=1, column=0, sticky=(tk.W, tk.E))
        
        # Bind mouse events
        self.canvas.bind("<Button-1>", self.on_canvas_click)
        self.canvas.bind("<Motion>", self.on_canvas_motion)
        # Bind mouse wheel zoom
        self.canvas.bind("<MouseWheel>", self.on_mousewheel)  # Windows
        self.canvas.bind("<Button-4>", self.on_mousewheel)     # Linux
        self.canvas.bind("<Button-5>", self.on_mousewheel)     # Linux
        self.canvas.focus_set()  # Make canvas focusable to receive wheel events
        
        # Coordinate input and display frame
        coord_frame = ttk.Frame(main_frame)
        coord_frame.grid(row=2, column=0, pady=(5, 2))
        
        ttk.Label(coord_frame, text="Coordinates:", font=('Arial', 10)).pack(side=tk.LEFT, padx=2)
        ttk.Label(coord_frame, text="X:", font=('Arial', 9)).pack(side=tk.LEFT, padx=2)
        self.x_entry = ttk.Entry(coord_frame, width=8, font=('Arial', 9))
        self.x_entry.pack(side=tk.LEFT, padx=2)
        self.x_entry.bind('<Return>', self.on_coord_entry)
        self.x_entry.bind('<KP_Enter>', self.on_coord_entry)
        
        ttk.Label(coord_frame, text="Y:", font=('Arial', 9)).pack(side=tk.LEFT, padx=2)
        self.y_entry = ttk.Entry(coord_frame, width=8, font=('Arial', 9))
        self.y_entry.pack(side=tk.LEFT, padx=2)
        self.y_entry.bind('<Return>', self.on_coord_entry)
        self.y_entry.bind('<KP_Enter>', self.on_coord_entry)
        
        ttk.Button(coord_frame, text="Go to", command=self.on_coord_entry, width=6).pack(side=tk.LEFT, padx=5)
        
        # Coordinate display label
        self.coord_label = ttk.Label(main_frame, text="Coordinates: Not selected", font=('Arial', 10))
        self.coord_label.grid(row=3, column=0, pady=(2, 2))
        
        # Zoom control frame
        zoom_frame = ttk.Frame(main_frame)
        zoom_frame.grid(row=4, column=0, pady=2)
        ttk.Label(zoom_frame, text="Zoom:", font=('Arial', 9)).pack(side=tk.LEFT, padx=2)
        ttk.Button(zoom_frame, text="Zoom In (+)", command=lambda: self.zoom_image(1.2)).pack(side=tk.LEFT, padx=2)
        ttk.Button(zoom_frame, text="Zoom Out (-)", command=lambda: self.zoom_image(0.8)).pack(side=tk.LEFT, padx=2)
        ttk.Button(zoom_frame, text="Reset", command=self.reset_zoom).pack(side=tk.LEFT, padx=2)
        self.zoom_label = ttk.Label(zoom_frame, text="100%", font=('Arial', 9))
        self.zoom_label.pack(side=tk.LEFT, padx=5)
        
        # Button frame
        button_frame = ttk.Frame(main_frame)
        button_frame.grid(row=5, column=0, pady=5)
        
        self.select_button = ttk.Button(button_frame, text="Confirm Selection & Process", 
                                        command=self.confirm_and_process, state='disabled')
        self.select_button.pack(side=tk.LEFT, padx=5)
        
        ttk.Button(button_frame, text="Reselect Coordinates", command=self.reset_selection).pack(side=tk.LEFT, padx=5)
        ttk.Button(button_frame, text="Select Folder", command=self.select_folder).pack(side=tk.LEFT, padx=5)
        ttk.Button(button_frame, text="Exit", command=self.root.quit).pack(side=tk.LEFT, padx=5)
        
        # Progress bar (determinate mode to show actual progress)
        self.progress = ttk.Progressbar(main_frame, mode='determinate')
        self.progress.grid(row=6, column=0, sticky=(tk.W, tk.E), pady=2)
        
        # Status label
        self.status_label = ttk.Label(main_frame, text="Loading first image...", font=('Arial', 9))
        self.status_label.grid(row=7, column=0, pady=2)
        
        # File path display label (clickable to copy)
        self.file_path_label = ttk.Label(main_frame, text="", font=('Arial', 8), 
                                         foreground='blue', cursor='hand2')
        self.file_path_label.grid(row=8, column=0, pady=1)
        self.file_path_label.bind("<Button-1>", self.copy_path_to_clipboard)
        
        # Open file button (initially hidden)
        self.open_file_button = ttk.Button(main_frame, text="Open File Location", 
                                          command=self.open_file_location, state='disabled')
        self.open_file_button.grid(row=9, column=0, pady=2)
        
        # Load first image
        self.load_first_image()
    
    def load_first_image(self):
        """Load the first DNG image for coordinate selection"""
        folder = Path(self.folder_path)
        dng_files = sorted(set(folder.glob('*.DNG')) | set(folder.glob('*.dng')))
        
        if not dng_files:
            messagebox.showerror("Error", f"No DNG files found in folder {self.folder_path}")
            self.root.quit()
            return
        
        try:
            self.status_label.config(text=f"Loading: {dng_files[0].name}...")
            self.root.update()
            
            # Read RAW image
            with rawpy.imread(str(dng_files[0])) as raw:
                # Get 16-bit RGB image first to obtain accurate dimensions (used for coordinate system)
                rgb_16bit = raw.postprocess(
                    use_camera_wb=True,      # Use camera white balance
                    half_size=False,         # Do not half the size
                    no_auto_bright=True,     # Do not auto-adjust brightness
                    output_bps=16,           # 16-bit to get dimensions
                    demosaic_algorithm=rawpy.DemosaicAlgorithm.AHD  # Use AHD algorithm
                )
                
                # Get dimensions of RGB image (this is the coordinate system used)
                self.original_height, self.original_width = rgb_16bit.shape[:2]
                
                # Convert to 8-bit for display (PIL Image requires 8-bit or special format)
                # Scale 16-bit values to 8-bit range (0-65535 -> 0-255)
                rgb_8bit = (rgb_16bit / 256).astype(np.uint8)
                
                # Convert to PIL Image
                self.original_image = Image.fromarray(rgb_8bit)
                
                # Calculate initial scaling factor to fit the window
                max_display_width = 1000
                max_display_height = 700
                
                img_width, img_height = self.original_image.size
                scale_w = max_display_width / img_width
                scale_h = max_display_height / img_height
                self.base_scale_factor = min(scale_w, scale_h, 1.0)  # Initial scaling factor
                self.scale_factor = self.base_scale_factor
                self.current_zoom = 1.0
                
                # Display image
                self.update_display_image()
                
                self.status_label.config(text=f"Loaded: {dng_files[0].name} ({img_width}x{img_height}) | "
                                             f"Use mouse wheel or buttons to zoom")
                
        except Exception as e:
            messagebox.showerror("Error", f"Error loading image: {str(e)}")
            self.root.quit()
    
    def on_canvas_click(self, event):
        """Handle canvas click event"""
        # Get canvas coordinates
        canvas_x = self.canvas.canvasx(event.x)
        canvas_y = self.canvas.canvasy(event.y)
        
        # Convert to original image coordinates
        original_x = int(canvas_x / self.scale_factor)
        original_y = int(canvas_y / self.scale_factor)
        
        # Ensure coordinates are within valid range (considering 2x2 region)
        if original_x < 0 or original_y < 0 or \
           original_x + 2 > self.original_width or original_y + 2 > self.original_height:
            messagebox.showwarning("Warning", 
                f"Coordinate ({original_x}, {original_y}) out of range or cannot fit a 2x2 region\n"
                f"Image dimensions: {self.original_width}x{self.original_height}")
            return
        
        self.selected_x = original_x
        self.selected_y = original_y
        
        # Update entry boxes
        self.x_entry.delete(0, tk.END)
        self.x_entry.insert(0, str(original_x))
        self.y_entry.delete(0, tk.END)
        self.y_entry.insert(0, str(original_y))
        
        # Update display
        self.update_coord_display()
        self.draw_selection_marker(canvas_x, canvas_y)
        self.select_button.config(state='normal')
    
    def on_canvas_motion(self, event):
        """Show coordinates when mouse moves"""
        canvas_x = self.canvas.canvasx(event.x)
        canvas_y = self.canvas.canvasy(event.y)
        
        original_x = int(canvas_x / self.scale_factor)
        original_y = int(canvas_y / self.scale_factor)
        
        if 0 <= original_x < self.original_width and 0 <= original_y < self.original_height:
            self.coord_label.config(text=f"Mouse position: ({original_x}, {original_y}) | "
                                        f"Selected: ({self.selected_x if self.selected_x is not None else 'None'}, "
                                        f"{self.selected_y if self.selected_y is not None else 'None'})")
    
    def update_display_image(self):
        """Update the displayed image"""
        if self.original_image is None:
            return
        
        img_width, img_height = self.original_image.size
        display_width = int(img_width * self.scale_factor)
        display_height = int(img_height * self.scale_factor)
        
        # Scale image for display
        self.display_image = self.original_image.resize(
            (display_width, display_height), Image.Resampling.LANCZOS)
        
        # Convert to PhotoImage
        self.photo = ImageTk.PhotoImage(self.display_image)
        
        # Clear canvas and redisplay
        self.canvas.delete("all")
        self.canvas.create_image(0, 0, anchor=tk.NW, image=self.photo)
        self.canvas.config(scrollregion=self.canvas.bbox("all"))
        
        # If a selection marker exists, redraw it
        if self.selected_x is not None and self.selected_y is not None:
            canvas_x = self.selected_x * self.scale_factor
            canvas_y = self.selected_y * self.scale_factor
            self.draw_selection_marker(canvas_x, canvas_y)
        
        # Update zoom label
        zoom_percent = int(self.current_zoom * 100)
        self.zoom_label.config(text=f"{zoom_percent}%")
    
    def zoom_image(self, zoom_factor, mouse_x=None, mouse_y=None):
        """Zoom the image, centering on mouse position"""
        if self.original_image is None:
            return
        
        # Limit zoom range (increased max zoom)
        min_zoom = 0.1
        max_zoom = 20.0  # Increased from 5.0 to 20.0
        
        new_zoom = self.current_zoom * zoom_factor
        new_zoom = max(min_zoom, min(max_zoom, new_zoom))
        
        if new_zoom == self.current_zoom:
            return  # No change, no need to update
        
        # If mouse position is provided, zoom centered on that point
        if mouse_x is not None and mouse_y is not None:
            # Save old scaling factor
            old_scale_factor = self.scale_factor
            
            # Get mouse position on canvas (considering current scroll)
            canvas_x = self.canvas.canvasx(mouse_x)
            canvas_y = self.canvas.canvasy(mouse_y)
            
            # Compute original image coordinates corresponding to mouse position (using old scaling)
            image_x = canvas_x / old_scale_factor
            image_y = canvas_y / old_scale_factor
            
            # Update scaling factor
            self.current_zoom = new_zoom
            self.scale_factor = self.base_scale_factor * self.current_zoom
            
            # Update display (this recreates the image and may reset scroll)
            self.update_display_image()
            
            # Wait for canvas to update
            self.root.update_idletasks()
            
            # Compute new canvas position for the same image coordinates
            new_canvas_x = image_x * self.scale_factor
            new_canvas_y = image_y * self.scale_factor
            
            # Get total size of canvas content
            bbox = self.canvas.bbox("all")
            if bbox:
                total_width = bbox[2] - bbox[0]
                total_height = bbox[3] - bbox[1]
                canvas_width = self.canvas.winfo_width()
                canvas_height = self.canvas.winfo_height()
                
                if total_width > 0 and total_height > 0 and canvas_width > 0 and canvas_height > 0:
                    # Compute scroll position needed
                    # We want new_canvas_x to appear at mouse_x
                    target_scroll_x = new_canvas_x - mouse_x
                    target_scroll_y = new_canvas_y - mouse_y
                    
                    # Limit scroll range
                    max_scroll_x = max(0, total_width - canvas_width)
                    max_scroll_y = max(0, total_height - canvas_height)
                    target_scroll_x = max(0, min(target_scroll_x, max_scroll_x))
                    target_scroll_y = max(0, min(target_scroll_y, max_scroll_y))
                    
                    # Apply new scroll position (convert to 0-1 ratio)
                    if total_width > canvas_width:
                        self.canvas.xview_moveto(target_scroll_x / total_width)
                    if total_height > canvas_height:
                        self.canvas.yview_moveto(target_scroll_y / total_height)
        else:
            # No mouse position, zoom centered on canvas center
            self.current_zoom = new_zoom
            self.scale_factor = self.base_scale_factor * self.current_zoom
            self.update_display_image()
    
    def reset_zoom(self):
        """Reset zoom to default"""
        if self.original_image is None:
            return
        
        self.current_zoom = 1.0
        self.scale_factor = self.base_scale_factor
        self.update_display_image()
    
    def on_mousewheel(self, event):
        """Handle mouse wheel zoom, centered on mouse position"""
        if self.original_image is None:
            return
        
        # Get mouse position on canvas
        mouse_x = event.x
        mouse_y = event.y
        
        # Windows uses delta, Linux uses num
        if event.num == 4 or event.delta > 0:
            self.zoom_image(1.1, mouse_x, mouse_y)  # Zoom in, centered on mouse
        elif event.num == 5 or event.delta < 0:
            self.zoom_image(0.9, mouse_x, mouse_y)  # Zoom out, centered on mouse
    
    def draw_selection_marker(self, canvas_x, canvas_y):
        """Draw selection marker on canvas"""
        # Clear previous marker
        self.canvas.delete("selection_marker")
        
        # Draw 2x2 region marker
        marker_size = 2 * self.scale_factor
        self.canvas.create_rectangle(
            canvas_x, canvas_y,
            canvas_x + marker_size, canvas_y + marker_size,
            outline='red', width=2, tags="selection_marker"
        )
        # Draw center point
        self.canvas.create_oval(
            canvas_x + marker_size/2 - 3, canvas_y + marker_size/2 - 3,
            canvas_x + marker_size/2 + 3, canvas_y + marker_size/2 + 3,
            fill='red', outline='red', tags="selection_marker"
        )
    
    def update_coord_display(self):
        """Update coordinate display"""
        if self.selected_x is not None and self.selected_y is not None:
            self.coord_label.config(
                text=f"Selected coordinates: ({self.selected_x}, {self.selected_y}) | "
                     f"2x2 region: ({self.selected_x}, {self.selected_y}) to "
                     f"({self.selected_x+1}, {self.selected_y+1})"
            )
    
    def reset_selection(self):
        """Reset selection"""
        self.selected_x = None
        self.selected_y = None
        self.canvas.delete("selection_marker")
        self.coord_label.config(text="Coordinates: Not selected")
        self.select_button.config(state='disabled')
        # Clear entry boxes
        self.x_entry.delete(0, tk.END)
        self.y_entry.delete(0, tk.END)
    
    def on_coord_entry(self, event=None):
        """Handle manual coordinate input"""
        if self.original_image is None:
            messagebox.showinfo("Info", "Please load an image first")
            return
        
        try:
            x_str = self.x_entry.get().strip()
            y_str = self.y_entry.get().strip()
            
            if not x_str or not y_str:
                messagebox.showwarning("Warning", "Please enter both X and Y coordinates")
                return
            
            x = int(x_str)
            y = int(y_str)
            
            # Check if coordinates are within valid range (considering 2x2 region)
            if x < 0 or y < 0 or x + 2 > self.original_width or y + 2 > self.original_height:
                messagebox.showwarning("Warning", 
                    f"Coordinates ({x}, {y}) out of range or cannot fit a 2x2 region\n"
                    f"Image dimensions: {self.original_width}x{self.original_height}\n"
                    f"Valid range: X: 0-{self.original_width-2}, Y: 0-{self.original_height-2}")
                return
            
            # Set selected coordinates
            self.selected_x = x
            self.selected_y = y
            
            # Update display
            self.update_coord_display()
            
            # Compute canvas coordinates and draw marker
            canvas_x = x * self.scale_factor
            canvas_y = y * self.scale_factor
            self.draw_selection_marker(canvas_x, canvas_y)
            
            # Scroll to selected position (if not visible)
            self.scroll_to_position(canvas_x, canvas_y)
            
            self.select_button.config(state='normal')
            
        except ValueError:
            messagebox.showerror("Error", "Please enter valid numeric coordinates")
    
    def scroll_to_position(self, canvas_x, canvas_y):
        """Scroll canvas to specified position"""
        # Get visible area of canvas
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        if canvas_width <= 1 or canvas_height <= 1:
            return  # Canvas not yet initialized
        
        # Get total size of canvas content
        bbox = self.canvas.bbox("all")
        if bbox is None:
            return
        
        total_width = bbox[2] - bbox[0]
        total_height = bbox[3] - bbox[1]
        
        if total_width <= 0 or total_height <= 0:
            return
        
        # Compute scroll distance to bring target to center
        scroll_x = canvas_x - canvas_width / 2
        scroll_y = canvas_y - canvas_height / 2
        
        # Limit scroll range
        scroll_x = max(0, min(scroll_x, total_width - canvas_width))
        scroll_y = max(0, min(scroll_y, total_height - canvas_height))
        
        # Scroll to position (convert to 0-1 ratio)
        if total_width > canvas_width:
            self.canvas.xview_moveto(scroll_x / total_width)
        if total_height > canvas_height:
            self.canvas.yview_moveto(scroll_y / total_height)
    
    def select_folder(self):
        """Select a folder containing DNG files"""
        folder = filedialog.askdirectory(
            title="Select folder containing DNG files",
            initialdir=self.folder_path if Path(self.folder_path).exists() else None
        )
        
        if folder:
            # Check if the folder contains DNG files
            folder_path = Path(folder)
            dng_files = sorted(set(folder_path.glob('*.DNG')) | set(folder_path.glob('*.dng')))
            
            if not dng_files:
                messagebox.showwarning("Warning", f"No DNG files found in folder {folder}")
                return
            
            # Update folder path
            self.folder_path = folder
            self.folder_path_label.config(text=folder)
            
            # Reset selection
            self.reset_selection()
            self.reset_zoom()
            
            # Reload first image
            self.load_first_image()
    
    def confirm_and_process(self):
        """Confirm selection and start processing"""
        if self.selected_x is None or self.selected_y is None:
            messagebox.showwarning("Warning", "Please select coordinates first")
            return
        
        # Disable button
        self.select_button.config(state='disabled')
        self.progress['value'] = 0
        self.progress['maximum'] = 100
        self.status_label.config(text="Processing all images, please wait...")
        
        # Get total file count to set progress bar maximum
        folder = Path(self.folder_path)
        dng_files = sorted(set(folder.glob('*.DNG')) | set(folder.glob('*.dng')))
        total_files = len(dng_files)
        
        if total_files == 0:
            messagebox.showerror("Error", "No DNG files found")
            self.select_button.config(state='normal')
            return
        
        # Progress update callback
        def update_progress(current, total, filename):
            """Update progress bar"""
            progress_value = int((current / total) * 100)
            self.root.after(0, lambda: self.progress.config(value=progress_value))
            self.root.after(0, lambda: self.status_label.config(
                text=f"Processing: {filename} ({current}/{total})"
            ))
        
        # Process in a new thread to avoid UI blocking
        def process_thread():
            try:
                output_path = process_image_folder(
                    self.folder_path, 
                    self.selected_x, 
                    self.selected_y, 
                    self.output_csv,
                    progress_callback=update_progress
                )
                self.root.after(0, lambda: self.on_processing_complete(output_path))
            except Exception as e:
                self.root.after(0, lambda: self.on_processing_error(str(e)))
        
        thread = threading.Thread(target=process_thread, daemon=True)
        thread.start()
    
    def on_processing_complete(self, output_path):
        """Processing completed"""
        self.progress['value'] = 100  # Set to 100% complete
        self.output_file_path = output_path
        
        if output_path and Path(output_path).exists():
            self.status_label.config(text="Processing complete! File saved")
            self.file_path_label.config(text=f"File location: {output_path} (click to copy path)")
            self.open_file_button.config(state='normal')
            
            messagebox.showinfo("Complete", 
                f"All images processed!\n\n"
                f"File saved to:\n{output_path}\n\n"
                f"Click 'Open File Location' button to view the file")
        else:
            self.status_label.config(text="Processing complete, but file path not found")
            messagebox.showwarning("Warning", "Processing complete but unable to locate saved file")
    
    def on_processing_error(self, error_msg):
        """Processing error"""
        self.progress['value'] = 0  # Reset progress bar
        self.status_label.config(text=f"Error: {error_msg}")
        messagebox.showerror("Error", f"Error processing images:\n{error_msg}")
        self.select_button.config(state='normal')
    
    def copy_path_to_clipboard(self, event=None):
        """Copy file path to clipboard"""
        if self.output_file_path:
            self.root.clipboard_clear()
            self.root.clipboard_append(self.output_file_path)
            self.status_label.config(text="Path copied to clipboard!")
    
    def open_file_location(self):
        """Open file location in file explorer"""
        if not self.output_file_path or not Path(self.output_file_path).exists():
            messagebox.showerror("Error", "File does not exist or path is invalid")
            return
        
        file_path = Path(self.output_file_path)
        system = platform.system()
        
        try:
            if system == "Windows":
                # Windows: open explorer and select the file
                subprocess.run(f'explorer /select,"{file_path}"', shell=True)
            elif system == "Darwin":  # macOS
                subprocess.run(["open", "-R", str(file_path)])
            else:  # Linux
                subprocess.run(["xdg-open", str(file_path.parent)])
        except Exception as e:
            messagebox.showerror("Error", f"Unable to open file location:\n{str(e)}")
    
    def run(self):
        """Run the GUI"""
        self.root.mainloop()

if __name__ == "__main__":
    # Configuration parameters
    DEFAULT_FOLDER = "处理"  # Default folder
    OUTPUT_CSV = "rgb_average_results.csv"  # Output filename
    
    # If default folder does not exist, let user choose
    if not Path(DEFAULT_FOLDER).exists():
        root = tk.Tk()
        root.withdraw()  # Hide main window
        
        folder = filedialog.askdirectory(
            title="Select folder containing DNG files",
            initialdir=Path.cwd()
        )
        
        root.destroy()
        
        if not folder:
            print("No folder selected, exiting")
            exit(0)
        
        FOLDER_PATH = folder
    else:
        FOLDER_PATH = DEFAULT_FOLDER
    
    # Launch GUI
    app = CoordinateSelector(FOLDER_PATH, OUTPUT_CSV)
    app.run()

Found 68 DNG files
Processing 2x2 region at (1271, 2294)...
------------------------------------------------------------
[1/68] Processing: IMG_0001.DNG
  ✓ Success
[2/68] Processing: IMG_0002.DNG
  ✓ Success
[3/68] Processing: IMG_0003.DNG
  ✓ Success
[4/68] Processing: IMG_0004.DNG
  ✓ Success
[5/68] Processing: IMG_0005.DNG
  ✓ Success
[6/68] Processing: IMG_0006.DNG
  ✓ Success
[7/68] Processing: IMG_0007.DNG
  ✓ Success
[8/68] Processing: IMG_0008.DNG
  ✓ Success
[9/68] Processing: IMG_0009.DNG
  ✓ Success
[10/68] Processing: IMG_0010.DNG
  ✓ Success
[11/68] Processing: IMG_0011.DNG
  ✓ Success
[12/68] Processing: IMG_0012.DNG
  ✓ Success
[13/68] Processing: IMG_0013.DNG
  ✓ Success
[14/68] Processing: IMG_0014.DNG
  ✓ Success
[15/68] Processing: IMG_0015.DNG
  ✓ Success
[16/68] Processing: IMG_0016.DNG
  ✓ Success
[17/68] Processing: IMG_0017.DNG
  ✓ Success
[18/68] Processing: IMG_0018.DNG
  ✓ Success
[19/68] Processing: IMG_0019.DNG
  ✓ Success
[20/68] Processing: IMG_0020.DNG
